### DEAM dataset - Database for Emotional Analysis of Music
[DEAM - Kaggle](https://www.kaggle.com/datasets/imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music)\
[DEAM - Université de Genève](https://cvml.unige.ch/databases/DEAM/)

Prepare data

In [1]:
# import sys
# import os

# # sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src")))
# sys.path.append(os.path.abspath(os.path.join("../src")))

# from audio_processing import (
#     load_mean_valence_arousal,
#     load_valence_arousal,
#     get_song_ids_and_labels,
#     DEAMSegmentGenerator,
# )

# print(load_mean_valence_arousal(os.path.abspath(os.path.join("../data/"))))

In [2]:
# print(load_valence_arousal(os.path.abspath(os.path.join("../data/"))))

In [3]:
# print(load_valence_arousal(os.path.abspath(os.path.join("../data/")), span=2))

Właściwe przygotowanie danych

In [4]:
# import tensorflow as tf

# print(tf.config.list_physical_devices("GPU"))

In [5]:
import tensorflow as tf

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
        tf.config.set_logical_device_configuration(
            gpus[0],
            [
                tf.config.LogicalDeviceConfiguration(memory_limit=10000)
            ],
        )
    except RuntimeError as e:
        print(e)

In [6]:
import sys
import os
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(os.path.join("../src")))

from model import MER_CNN_Model, MER_CNN_Simple, MER_CNN_VGG_Style, MER_CRNN, MER_CNN_MobileNet
from audio_processing import DEAMSegmentGenerator, preprocess_deam, load_valence_arousal, get_song_ids_and_labels

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [7]:
# PARAMETRY GLOBALNE - MUSZĄ BYĆ SPÓJNE W CAŁYM PROJEKCIE
SR = 22050  # Domyślna częstotliwość librosa.load
HOP_LENGTH = 512  # Krok przesunięcia okna FFT
SEGMENT_WIDTH = 128  # Szerokość obrazka wejściowego do sieci (oś czasu)
N_MELS = 128  # Wysokość obrazka wejściowego do sieci (oś częstotliwości)
SPAN = 0.5  # Jak bardzo uśredniamy etykiety w DataFrame (w sekundach)
DEAM_DIR = os.path.abspath(os.path.join("../data/"))
AUDIO_DIR = os.path.abspath(os.path.join("../data/DEAM/MEMD_audio/"))
AUDIO_NPY_DIR = os.path.abspath(os.path.join("../data/DEAM/audio_npy/"))

In [8]:
# labels_df = load_valence_arousal(DEAM_DIR, span=SPAN)
# dynamic_labels = get_song_ids_and_labels(labels_df)

In [9]:

labels_df = load_valence_arousal(DEAM_DIR, span=0.5)
dynamic_labels = get_song_ids_and_labels(labels_df)

print(labels_df.columns[:10])
print("Songs in labels:", len(dynamic_labels))
print("Example song labels count:", len(next(iter(dynamic_labels.values()))))

Index(['song_id', 'sample_15000ms_x', 'sample_15500ms_x', 'sample_16000ms_x',
       'sample_16500ms_x', 'sample_17000ms_x', 'sample_17500ms_x',
       'sample_18000ms_x', 'sample_18500ms_x', 'sample_19000ms_x'],
      dtype='object')
Songs in labels: 1744
Example song labels count: 0


In [10]:
# Run only once to preprocess audio files into Mel spectrograms and save as .npy
# preprocess_deam(AUDIO_DIR, AUDIO_NPY_DIR, n_mels=N_MELS, hop_length=HOP_LENGTH, sr=SR)

In [11]:
existing_song_ids = [
    s_id for s_id in dynamic_labels.keys() if os.path.exists(os.path.join(AUDIO_NPY_DIR, f"{s_id}.npy"))
]

print(f"Znaleziono {len(existing_song_ids)} utworów z danymi audio i etykietami.")

Znaleziono 1744 utworów z danymi audio i etykietami.


In [12]:
train_ids, val_ids = train_test_split(existing_song_ids, test_size=0.2)
print(f"Trening na {len(train_ids)} utworach, walidacja na {len(val_ids)} utworach.")

Trening na 1395 utworach, walidacja na 349 utworach.


In [13]:
train_generator = DEAMSegmentGenerator(
    song_ids=train_ids,
    labels_dict=dynamic_labels,
    data_dir=AUDIO_NPY_DIR,
    segment_width=SEGMENT_WIDTH,
    hop_size=HOP_LENGTH,
    sr=SR,
    batch_size=32,
    shuffle=True,
)
valid_generator = DEAMSegmentGenerator(
    song_ids=val_ids,
    labels_dict=dynamic_labels,
    data_dir=AUDIO_NPY_DIR,
    segment_width=SEGMENT_WIDTH,
    hop_size=HOP_LENGTH,
    sr=SR,
    batch_size=32,
    shuffle=False,
)

In [14]:
model_CNN = MER_CNN_Simple()
model_CNN.fit(
    train_generator,
    validation_generator=valid_generator,
    epochs=50,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
        ModelCheckpoint("best_model.h5", save_best_only=True),
    ],
)

c:\Users\goodm\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: min() arg is an empty sequence

In [ ]:
from datetime import datetime
import pickle
import glob


def save_model(model, save_path, history, name=""):
    save_dir = "{}/".format(save_path)
    os.makedirs(save_dir, exist_ok=True)

    file_details = (
        f"{model.__class__.__name__}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{name if name else ''}.h5"
    )
    model_path = os.path.join(save_dir, file_details)
    model.save(model_path)
    print(f"Model saved to: {model_path}")
    if history is not None:
        pickle.dump(history, open(f"{save_dir}/history_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.pkl", "wb"))


def load_model(model_path, history):
    model_files = glob.glob(model_path + "/*.h5")
    latest_model_file = max(model_files, key=os.path.getctime)
    model = tf.keras.models.load_model(latest_model_file)
    history_files = glob.glob(model_path + "/history*.pkl")
    latest_history_file = max(history_files, key=os.path.getctime) if history_files else None
    history = pickle.load(open(latest_history_file, "rb")) if history else None
    return model, history

In [ ]:
save_model(model_CNN._model, "models/MER_CNN_Simple/", history=model_CNN._model.history, name="MER_CNN_Simple")

In [ ]:
model123, history123 = load_model("models/MER_CNN_Simple/", history=True)

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Wykres straty (Loss - MSE)
    ax[0].plot(history.history["loss"], label="Train Loss")
    ax[0].plot(history.history["val_loss"], label="Val Loss")
    ax[0].set_title("Model Loss (MSE)")
    ax[0].legend()

    # Wykres błędu (MAE)
    ax[1].plot(history.history["mae"], label="Train MAE")
    ax[1].plot(history.history["val_mae"], label="Val MAE")
    ax[1].set_title("Model Metric (MAE)")
    ax[1].legend()

    plt.show()

In [ ]:
plot_history(history123)

In [ ]:
def plot_va_scatter(y_true, y_pred):
    plt.figure(figsize=(8, 8))
    plt.scatter(y_true[:, 0], y_true[:, 1], alpha=0.3, label="Prawdziwe", color="blue")
    plt.scatter(y_pred[:, 0], y_pred[:, 1], alpha=0.3, label="Przewidziane", color="red")

    # Rysowanie osi V-A
    plt.axhline(0, color="black", lw=1)
    plt.axvline(0, color="black", lw=1)

    plt.xlabel("Valence")
    plt.ylabel("Arousal")
    plt.title("Rozkład przewidywań w przestrzeni V-A")
    plt.legend()
    plt.show()

In [ ]:
import numpy as np

valid_generator.shuffle = False
valid_generator.on_epoch_end()  # resetujemy kolejność próbek

y_pred = model123.predict(valid_generator)


def get_y_true(generator):
    y_true = []

    # Iterujemy tylko po pełnych batchach (tyle samo, ile widzi model.predict)
    num_batches = len(generator)
    for i in range(num_batches):
        # Pobieramy batch danych (X, y)
        _, y_batch = generator[i]
        y_true.extend(y_batch)

    return np.array(y_true)


y_true = get_y_true(valid_generator)
y_pred = model123.predict(valid_generator)
plot_va_scatter(y_true, y_pred)

In [ ]:
from tensorflow.keras.models import Model


def visualize_feature_maps(model, sample_spectrogram, layer_name):
    # Tworzymy model pośredni, który kończy się na wybranej warstwie
    intermediate_model = Model(inputs=model.input, outputs=model.get_layer(layer_name).output)
    feature_maps = intermediate_model.predict(sample_spectrogram[np.newaxis, ...])

    # Wyświetlamy np. pierwsze 8 filtrów
    num_filters = 8
    fig, axes = plt.subplots(1, num_filters, figsize=(20, 5))
    for i in range(num_filters):
        axes[i].imshow(feature_maps[0, :, :, i], cmap="viridis")
        axes[i].axis("off")
    plt.show()


import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf


def visualize_feature_maps(model, sample_spectrogram, layer_name):
    # 1. Znajdujemy indeks warstwy o danej nazwie
    layer_idx = None
    for i, layer in enumerate(model.layers):
        if layer.name == layer_name:
            layer_idx = i
            break

    if layer_idx is None:
        print(f"Błąd: Nie znaleziono warstwy o nazwie '{layer_name}'.")
        print("Dostępne warstwy:", [l.name for l in model.layers])
        return

    # 2. Tworzymy nowy model Sequential, który zawiera warstwy od 0 do layer_idx
    # To omija błąd AttributeError: sequential has never been called
    intermediate_model = tf.keras.Sequential(model.layers[: layer_idx + 1])

    # 3. Przygotowujemy dane (dodajemy wymiar batcha: (1, 128, 128, 1))
    input_data = sample_spectrogram[np.newaxis, ...]

    # 4. Przewidujemy wynik dla tej warstwy
    feature_maps = intermediate_model.predict(input_data)

    # 5. Wizualizacja (np. pierwsze 8 filtrów)
    num_filters = min(8, feature_maps.shape[-1])
    fig, axes = plt.subplots(1, num_filters, figsize=(20, 5))

    if num_filters == 1:
        axes = [axes]  # obsługa przypadku z jednym filtrem

    for i in range(num_filters):
        # Wyświetlamy mapę cech (feature map)
        axes[i].imshow(feature_maps[0, :, :, i], cmap="viridis")
        axes[i].set_title(f"Filtr {i}")
        axes[i].axis("off")

    plt.suptitle(f"Mapy cech dla warstwy: {layer_name}")
    plt.show()

In [ ]:
visualize_feature_maps(model123, valid_generator[0][0][0], layer_name="conv2d_1")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os


def plot_song_evolution(model, generator, song_id):
    song_id = str(song_id)

    # 1. Znajdź wszystkie próbki dla tej piosenki i POSORTUJ je chronologicznie
    # Musimy to posortować, bo generator robi shuffle!
    song_samples = [s for s in generator.samples if s[0] == song_id]
    song_samples.sort(key=lambda x: x[1])  # sortowanie po start_frame

    if not song_samples:
        print(f"Błąd: Nie znaleziono próbek dla piosenki {song_id} w tym generatorze.")
        return

    X_song = []
    y_true_song = []
    timestamps = []

    # 2. Załaduj spektrogram całego utworu raz (oszczędność czasu)
    file_path = os.path.join(generator.data_dir, f"{song_id}.npy")
    full_spec = np.load(file_path, mmap_mode="r")

    # 3. Pobierz segmenty i etykiety
    for _, start_frame in song_samples:
        # Wycinek audio
        segment = full_spec[:, start_frame : start_frame + generator.segment_width]
        X_song.append(segment)

        # Przelicz klatkę na sekundy dla osi X
        time_sec = (start_frame * generator.hop_size) / generator.sr
        timestamps.append(time_sec)

        # Pobierz etykietę (znajdź najbliższą czasowo w słowniku)
        time_ms = int(time_sec * 1000)
        available_times = list(generator.labels[song_id].keys())
        closest_time = min(available_times, key=lambda x: abs(x - time_ms))
        y_true_song.append(generator.labels[song_id][closest_time])

    # 4. Predykcja modelu (cała piosenka na raz)
    X_song = np.array(X_song)[..., np.newaxis]
    y_pred_song = model.predict(X_song)
    y_true_song = np.array(y_true_song)

    # 5. Rysowanie wykresu
    plt.figure(figsize=(15, 6))

    # Valence
    plt.plot(timestamps, y_true_song[:, 0], "b-", alpha=0.3, label="Valence (Prawda)")
    plt.plot(timestamps, y_pred_song[:, 0], "b--", lw=2, label="Valence (Model)")

    # Arousal
    plt.plot(timestamps, y_true_song[:, 1], "r-", alpha=0.3, label="Arousal (Prawda)")
    plt.plot(timestamps, y_pred_song[:, 1], "r--", lw=2, label="Arousal (Model)")

    plt.title(f"Analiza emocji w czasie - Utwór ID: {song_id}")
    plt.xlabel("Czas (sekundy)")
    plt.ylabel("Wartość (-1 do 1)")
    plt.legend(loc="upper right", bbox_to_anchor=(1.15, 1))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# PRZYKŁAD UŻYCIA:
# Wybierz ID piosenki z tych, które są w Twoim zbiorze walidacyjnym
random_song_id = val_ids[0]
plot_song_evolution(model123, valid_generator, random_song_id)

#### inne

In [ ]:
from evaluator import EmotionEvaluator

In [ ]:
evaluator = EmotionEvaluator()
results = evaluator.evaluate(
    true_valences=y_true[:, 0], true_arousals=y_true[:, 1], pred_valences=y_pred[:, 0], pred_arousals=y_pred[:, 1]
)

In [ ]:
os.makedirs("results", exist_ok=True)

evaluator.plot_confusion_matrix(results, save_path="results/emotion_confusion_matrix.png")
evaluator.plot_distributions(results, save_path="results/emotion_distributions.png")
evaluator.plot_valence_arousal_scatter(
    true_valences=y_true[:, 0],
    true_arousals=y_true[:, 1],
    pred_valences=y_pred[:, 0],
    pred_arousals=y_pred[:, 1],
    save_path="results/emotion_scatter.png",
)